In [1]:
import pandas as pd
import numpy as np

# Models
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("data/transformed_train.csv")

# -----------------------------
# TARGET
# -----------------------------
target = "Demand"

# -----------------------------
# DROP LEAKAGE / DUPLICATES
# -----------------------------
drop_cols = [
    "Units Sold",
    "Inventory Level",
    "lag_inventory_1",
    "roll_inventory_mean_7"
]

df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# -----------------------------
# FEATURES & TARGET
# -----------------------------
X = df.drop(columns=[target])
y = df[target]

# -----------------------------
# TIME SPLIT (80-20)
# -----------------------------
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# -----------------------------
# RANDOM FOREST MODEL
# -----------------------------
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

# -----------------------------
# XGBOOST MODEL
# -----------------------------
xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train, y_train)

# -----------------------------
# PREDICTIONS
# -----------------------------
rf_pred = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)

# Ensemble prediction
ensemble_pred = (rf_pred + xgb_pred) / 2

# -----------------------------
# METRICS FUNCTION
# -----------------------------
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    total_abs_error = np.sum(np.abs(y_true - y_pred))
    total_sq_error = np.sum((y_true - y_pred) ** 2)

    return mae, rmse, r2, total_abs_error, total_sq_error

# -----------------------------
# EVALUATE MODELS
# -----------------------------
models = {
    "Random Forest": rf_pred,
    "XGBoost": xgb_pred,
    "Ensemble": ensemble_pred
}

metrics_list = []

for name, pred in models.items():
    mae, rmse, r2, total_abs, total_sq = evaluate(y_test, pred)

    metrics_list.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Total_Abs_Error": total_abs,
        "Total_Sq_Error": total_sq
    })

    print(f"\n{name}")
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R2:", round(r2, 4))
    print("Total Abs Error:", round(total_abs, 2))
    print("Total Sq Error:", round(total_sq, 2))

# -----------------------------
# SAVE FORECAST RESULTS
# -----------------------------
forecast_results = pd.DataFrame({
    "Actual_Demand": y_test.values,
    "RF_Pred": rf_pred,
    "XGB_Pred": xgb_pred,
    "Ensemble_Pred": ensemble_pred
})

forecast_results.to_csv("forecast_results.csv", index=False)

# -----------------------------
# SAVE MODEL METRICS
# -----------------------------
metrics_df = pd.DataFrame(metrics_list)

metrics_df.to_csv("model_metrics.csv", index=False)

# -----------------------------
# FINISHED
# -----------------------------
print("\nFiles saved successfully:")
print("1. forecast_results.csv")
print("2. model_metrics.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/transformed_train.csv'